# Baseline Models — F1 Race Position Prediction

Establishes two lower-bound benchmarks that every subsequent model must beat:

| Baseline | Description |
|---|---|
| **GridPosition** | Rule-based: predict finishing position = starting grid position |
| **DummyRegressor** | Statistical floor: always predict the training-set mean |

Metrics used throughout the project:
- **MAE** (Mean Absolute Error) — average position error
- **Spearman ρ** — rank-order correlation between predicted and actual finishing positions
- **Macro F1** — treats each position (1–20) as a class; averages F1 equally across all positions regardless of frequency

In [ ]:
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, f1_score
from scipy.stats import spearmanr

## 2. Load Data

Train/test splits were produced by the feature engineering pipeline and persisted as Parquet files under `data/`.
The split is temporal — training uses seasons prior to the test year to avoid leakage.

## 1. Setup

In [ ]:
train = pd.read_parquet('../../data/train.parquet')

In [ ]:
test = pd.read_parquet('../../data/test.parquet')

## 3. Prepare Features and Target

`RacePosition` (1–20) is the regression target. All other columns are features.

In [ ]:
test_Y = test['RacePosition']
train_Y = train['RacePosition']

In [ ]:
test_X = test.drop(columns=['RacePosition'])
train_X = train.drop(columns=['RacePosition'])

## 4. Validate Data

Confirm no missing values before training — the feature engineering pipeline should have handled imputation upstream.

In [ ]:
for name, df in [("train_X", train_X), ("train_Y", train_Y), ("test_X", test_X), ("test_Y", test_Y)]:
    n_null = df.isna().sum().sum() if hasattr(df, "columns") else df.isna().sum()
    status = "OK" if n_null == 0 else f"WARNING: {n_null} nulls"
    print(f"{name}: {status}")

## 5. Baseline Models

Two baselines are evaluated:

- **GridPosition**: assumes the driver finishes where they started — a strong real-world heuristic in F1.
- **DummyRegressor (mean)**: always predicts the average finishing position — the absolute statistical floor.

Any trained model must beat both to be worth deploying.

In [ ]:
def macro_f1(y_true, y_pred):
    """Round continuous predictions to nearest integer position, then compute macro F1."""
    y_pred_int = pd.Series(y_pred).round().clip(1, 20).astype(int)
    return f1_score(y_true.astype(int), y_pred_int, average="macro", zero_division=0)

def safe_spearmanr(y_true, y_pred):
    """Return 0.0 when predictions are constant (ρ is undefined for constant input)."""
    if pd.Series(y_pred).nunique() == 1:
        return 0.0
    return spearmanr(y_true, y_pred).statistic

results = {}

# Rule-based: grid position as prediction
grid_preds = test_X["GridPosition"]
results["GridPosition"] = {
    "MAE": mean_absolute_error(test_Y, grid_preds),
    "Spearman_rho": safe_spearmanr(test_Y, grid_preds),
    "Macro_F1": macro_f1(test_Y, grid_preds),
}

# Statistical floor: predict training-set mean
dummy = DummyRegressor(strategy="mean")
dummy.fit(train_X, train_Y)
dummy_preds = dummy.predict(test_X)
results["DummyRegressor"] = {
    "MAE": mean_absolute_error(test_Y, dummy_preds),
    "Spearman_rho": safe_spearmanr(test_Y, dummy_preds),
    "Macro_F1": macro_f1(test_Y, dummy_preds),
}

print(f"{'Model':<20} {'MAE':>6}  {'Spearman ρ':>10}  {'Macro F1':>8}")
print("-" * 52)
for model, metrics in results.items():
    print(f"{model:<20} {metrics['MAE']:>6.2f}  {metrics['Spearman_rho']:>10.3f}  {metrics['Macro_F1']:>8.3f}")

In [ ]:
results_df = pd.DataFrame(results).T.rename_axis("model").reset_index()
results_df.to_csv("../../reports/baseline_results.csv", index=False)
results_df

## 6. Takeaways

The GridPosition baseline is surprisingly competitive (Spearman ρ ≈ 0.65) because grid position strongly correlates with race outcome in F1. This sets a meaningful bar: a trained model needs to **beat MAE ≈ 10.4, ρ ≈ 0.65, and Macro F1 ≈ 0.05** to add value over a rule-based heuristic.

The DummyRegressor MAE (~5) is lower because it predicts near the middle of the 1–20 range — but its Spearman ρ and Macro F1 are near 0 (no ranking or classification power). A useful model must beat _both_ baselines on _all three_ metrics.

**Note on Macro F1 for regression**: continuous predictions are rounded to the nearest integer position (1–20) before scoring. Macro averaging weights all 20 position classes equally, making it sensitive to accuracy on rare positions (e.g. exact top-3 or bottom-3 finishes).